In [0]:
%sql
-- ============================================================
-- 01_create_materialized_views
-- Modelo Dimensional (Star Schema) sobre tablas Silver y Gold
-- Este script debe ejecutarse sobre el SQL Warehouse.
-- ============================================================

USE CATALOG fintech_finpay;

In [0]:
%sql
CREATE OR REPLACE MATERIALIZED VIEW gold.fact_transactions
COMMENT 'Tabla de hechos central combinando transacciones y KPIs de riesgo'
AS
SELECT 
    t.transaction_id,
    t.user_id,
    t.merchant_id,
    t.channel,
    t.transaction_type,
    t.amount,
    t.currency,
    t.transaction_date,
    t.status,
    t.reference_id,
    k.tasa_reversa,
    k.score_riesgo
FROM silver.transactions t
LEFT JOIN gold.risk_kpis k 
    ON t.merchant_id = k.merchant_id 
    AND t.channel = k.channel 
    AND t.transaction_date = k.fecha;

In [0]:
%sql
CREATE OR REPLACE MATERIALIZED VIEW gold.dim_merchant
COMMENT 'Dimensión de comercios'
AS
SELECT 
    merchant_id,
    merchant_name,
    category,
    country,
    affiliation_date,
    status,
    risk_level
FROM silver.merchants;

-- ---------------------------------------------------------

CREATE OR REPLACE MATERIALIZED VIEW gold.dim_user
COMMENT 'Dimensión de usuarios (PII protegido por Unity Catalog)'
AS
SELECT 
    user_id,
    full_name,
    document_id,
    email,
    phone,
    country,
    segment,
    registration_date
FROM silver.users;

-- ---------------------------------------------------------

CREATE OR REPLACE MATERIALIZED VIEW gold.dim_channel
COMMENT 'Dimensión descriptiva de canales'
AS
SELECT DISTINCT 
    channel AS channel_id,
    CASE channel
        WHEN 'web' THEN 'Portal Web'
        WHEN 'app' THEN 'Aplicación Móvil'
        WHEN 'pos' THEN 'Punto de Venta Físico'
    END AS channel_name
FROM silver.transactions;

In [0]:
%sql
CREATE OR REPLACE MATERIALIZED VIEW gold.dim_date
COMMENT 'Dimensión calendario generada dinámicamente sin huecos de fechas'
AS
SELECT
    CAST(DATE_FORMAT(d.fecha, 'yyyyMMdd') AS INT) AS date_key,
    d.fecha                                       AS full_date,
    YEAR(d.fecha)                                 AS anio,
    QUARTER(d.fecha)                              AS trimestre,
    MONTH(d.fecha)                                AS mes,
    DATE_FORMAT(d.fecha, 'MMMM')                  AS nombre_mes,
    WEEKOFYEAR(d.fecha)                           AS semana_anio,
    DAYOFMONTH(d.fecha)                           AS dia_mes,
    DATE_FORMAT(d.fecha, 'EEEE')                  AS nombre_dia,
    CASE WHEN DAYOFWEEK(d.fecha) IN (1, 7) THEN TRUE ELSE FALSE END AS es_fin_semana
FROM (
    SELECT EXPLODE(
        SEQUENCE(
            CAST('2023-01-01' AS DATE), 
            CAST('2030-12-31' AS DATE), 
            INTERVAL 1 DAY
        )
    ) AS fecha
) d;

In [0]:
%sql
-- Verificar que las vistas se crearon correctamente
SHOW MATERIALIZED VIEWS IN gold;